# Visual Computing: Object Tracking and Motion Analysis
 In this coursework, you will implement various object tracking algorithms and motion analysis techniques using computer vision. The goal is to understand and apply different tracking approaches, analyse their performance, and evaluate their effectiveness under real-world conditions. This coursework is worth 7% of the total marks for the unit.

### Change Detection Using GMM [10%]
#### Objective:
Implement Change Detection using a Gaussian Mixture Model (GMM) for object tracking from scratch. Follow the algorithm outlined in the provided lecture slide for modelling pixel changes.
cv2.createBackgroundSubtractorMOG2, is not allowed.
#### Implementation Details:
Model the image as a mixture of Gaussians and classify each pixel as foreground or background. This will be done based on pixel probability distributions, as explained in the lecture. Update the Gaussian distributions per pixel to decide whether a pixel belongs to the background or foreground. Evaluate the effectiveness of the model in classifying moving objects. You can use cv2.VideoCapture(), cv2.cvtColor(),cv2.COLOR_BGR2GRAY(), cv2.waitKey().


In [1]:
# Gaussian Mixture Model (GMM) for Change Detection
# -------------------------------------------------
# Instructions: Below, I provide a code structure as a guidance to get you started. Of course you can use your own code structure to achieve the objective. Your marks will NOT be deducted if you use your own code structure :) 
# Complete the sections marked with '### ENTER YOUR CODE HERE ###' to implement 
# the Gaussian Mixture Model for background subtraction and change detection.

import cv2
import numpy as np

# Initialize the GMMBackgroundSubtractor Class
class GMMBackgroundSubtractor:
    def __init__(self, frame_shape, num_gaussians=4, learning_rate=0.01, threshold=2.5):
        """
        Initialize Gaussian parameters: means, variances, and weights.

        Args:
            frame_shape (tuple): Shape of the input frame (height, width).
            num_gaussians (int): Number of Gaussian models per pixel.
            learning_rate (float): Rate at which the model updates.
            threshold (float): Threshold for matching a pixel to a Gaussian.
        """
        self.num_gaussians = num_gaussians
        self.learning_rate = learning_rate
        self.threshold = threshold
        
        # Initialize the means, variances, and weights for each Gaussian
        self.frame_shape = frame_shape
        self.means = np.random.rand(self.frame_shape[0], self.frame_shape[1], self.num_gaussians) * 255
        self.variances = np.ones((self.frame_shape[0], self.frame_shape[1], self.num_gaussians)) * 25 #uses 15 as it is a good starting point for the variance
        self.weights = np.ones((self.frame_shape[0], self.frame_shape[1], self.num_gaussians)) / self.num_gaussians
       

    def apply(self, frame):
        """
        Apply GMM to detect foreground objects.

        Args:
            frame (ndarray): Input video frame.

        Returns:
            foreground_mask (ndarray): Binary mask indicating foreground pixels.
        """
        
        """"""
        # Convert frame to grayscale if it isn't already
        if len(frame.shape) == 3:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        frame = frame.astype(np.float32)
        fg_mask = np.zeros((self.frame_shape[0], self.frame_shape[1]), dtype=np.uint8)
        
        # Calculate the absolute difference between the frame and Gaussian means
        abs_differences = np.zeros((self.frame_shape[0], self.frame_shape[1], self.num_gaussians))
        for i in range(self.num_gaussians):
            abs_differences[:, :, i] = np.abs(self.means[:, :, i] - frame)
        
        matches = abs_differences < (self.threshold * np.sqrt(self.variances))
        
        background_mask = np.sum(self.weights * matches / self.variances, axis=-1) > 0
        fg_mask = np.where(background_mask, 0, 255)
        
        
        """
        # Check which pixels match any of the Gaussians
        smallest = np.argmin(abs_differences, axis=2)
        matches = abs_differences < (self.threshold * np.sqrt(self.variances))
        for i in range(self.frame_shape[0]):
            for j in range(self.frame_shape[1]):
                if abs_differences[i, j, smallest[i, j]] < 2.5 * np.sqrt(self.variances[i, j, smallest[i, j]]):
                    matches[i, j, smallest[i, j]] = True
                    if self.weights[i, j, smallest[i, j]] / np.sqrt(self.variances[i, j, smallest[i, j]]) >= 0.5:
                        fg_mask[i, j] = 255
                    else:
                        fg_mask[i, j] = 0
                    
        """
                    

        # Step 1: Update matched Gaussians (means, variances, weights)
        alpha = self.learning_rate

        # Vectorized update of means and variances
        self.weights = (1 - alpha) * self.weights + alpha * matches
        self.means = (1 - alpha) * self.means + alpha * frame[:, :, None] * matches
        diff = frame[:, :, None] - self.means
        self.variances = (1 - alpha) * self.variances + alpha * (diff ** 2) * matches
        
        # Normalize weights so they sum to 1
        self.weights /= np.sum(self.weights, axis=-1, keepdims=True)
        print(np.sum(self.weights, axis=-1))

        ### ENTER YOUR CODE HERE ###
        # Step 2: Classify Foreground
      
        
        return fg_mask.astype(np.uint8)

# ------------------- Main Code to Run the GMM ------------------- #

video_path = '/Users/edwardpetcher/Downloads/Tracking_model_inputs_outputs_extra_tests/1. GMM Model_input_output/GMM_input-cars.mp4'  # Replace with your video file
cap = cv2.VideoCapture(video_path)

# Read the first frame to get the frame shape
ret, first_frame = cap.read()
if not ret:
    print("Error: Couldn't read the video.")
    cap.release()
    exit()

frame_shape = first_frame.shape[:2]  # (height, width)

# Initialize GMM Background Subtractor
gmm_subtractor = GMMBackgroundSubtractor(frame_shape)

while True:
    # Read a frame
    ret, frame = cap.read()
    if not ret:
        break  # End of video

    # Apply GMM background subtraction
    fg_mask = gmm_subtractor.apply(frame)

    # Display results
    cv2.imshow("Original Frame", frame)
    cv2.imshow("Foreground Mask", fg_mask)

    # Press 'q' to exit
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()


Error: Couldn't read the video.


OpenCV: Couldn't read video stream from file "input_video.mp4"


AttributeError: 'NoneType' object has no attribute 'shape'

: 

### Custom Lucas-Kanade (with OpenCV GMM) [10%]
#### Objective
In this task, students will be using Lucas-Kanade Optical Flow to track detected moving objects. You can use the OpenCV GMM cv2.createBackgroundSubtractorMOG2() for this section.  
#### Implementation Details:
Detect foreground objects using the OpenCV GMM, then extract feature points inside the foreground mask. Thereafter, implement and apply your custom Lucas-Kanade Optical Flow to track moving objects frame-by-frame. Visualise this via Motion vectors which are optical flow arrows showing direction and magnitude of object movement. You can use cv2.Sobel(), cv2.VideoCapture(), cv2.goodFeaturesToTrack(), cv2.cvtColor(),cv2.COLOR_BGR2GRAY(), cv2.waitKey(), and for visualization cv2.circle(),cv2.arrowedLine. 


In [ ]:
import cv2
import numpy as np

class LucasKanadeTracker:
    def __init__(self):
        pass

    def detect_features(self, frame, mask):
        ### ENTER YOUR CODE HERE ###
        pass

    def calculate_optical_flow(self, prev_frame, curr_frame, prev_points):
        ### ENTER YOUR CODE HERE ###
        pass

    def draw_motion_vectors(self, frame, prev_points, new_points, status):
        ### ENTER YOUR CODE HERE ###
        pass

# Load Video
cap = cv2.VideoCapture('input_video.mp4')  # Replace with your video file

### ENTER YOUR CODE TO PROCESS THE VIDEO HERE ###


cap.release()
cv2.destroyAllWindows()


### Template Matching for Object Tracking [10%] 
#### Objective:
Students will implement template matching using a weighted histogram matching technique from scratch to track a target object across multiple frames in a video. Experiment with NCC to obtain the results.
#### Implementation Details:
User manually selects a target object in the first frame. Apply template matching to locate the object in subsequent frames. Experiment with the similarity metric:  NCC (Normalized Cross-Correlation) between template & search region. Draw a bounding box around the detected object in each frame (The best match is marked with a rectangle in each frame). You can use cv2.matchTemplate(),cv2.selectROI(),cv2.minMaxLoc (),cv2.TM_CCORR_NORMED (),cv2.COLOR_BGR2GRAY (), cv2.VideoCapture(), cv2.cvtColor(), cv2.waitKey(),cv2.normalize().



In [ ]:
import cv2
import numpy as np

class TemplateMatcher:
    def __init__(self):
        pass

    def select_template(self, frame):
        """
        Allow the user to manually select the target object in the first frame.
        """
        ### ENTER YOUR CODE HERE ###
        # Use cv2.selectROI() to select the region of interest (ROI).
        pass

    def match_template(self, frame, template):
        """
        Apply template matching using Normalized Cross-Correlation (NCC).
        """
        ### ENTER YOUR CODE HERE ###
     
        pass

    def draw_bounding_box(self, frame, top_left, template_size):
        """
        Draw a bounding box around the detected object.
        """
        ### ENTER YOUR CODE HERE ###
        # Use cv2.rectangle() to draw the box.
        pass

# Load Video
cap = cv2.VideoCapture('input_video.mp4')  # Replace with your video file

### ENTER YOUR CODE TO PROCESS THE VIDEO HERE ###


cap.release()
cv2.destroyAllWindows()


### Improving Template Matching for Object Tracking [5% + 10%]
#### Objective:
After implementing template matching, students should explore potential improvements.  The key challenges with template matching are: 1. Scale changes 2. Rotation Changes 3. Brightness/ contrast changes and Occlusions.
#### Implementation Details:
[A] Multi-Scale Template Matching (Handling Scale Changes)
Instead of using a fixed-size template, track multiple scales: Resize the template to 3 different scales (of your choice) before matching. Match at different pyramid levels using cv2.pyrDown() and cv2.pyrUp(). You can use prebuilt functions: cv2.matchTemplate(),cv2.minMaxLoc(),cv2.resize().

[B] Rotation-Invariant Matching
Rotate the template at 3 different angles (of your choice)  and match each rotated version. You can use cv2.getRotationMatrix2D(),cv2.warpAffine().

[C] Using Feature-Based Matching Instead of Pixel-Based Matching
Template matching relies on raw pixel values, making it sensitive to changes. Instead of comparing pixel intensities, extract SIFT features and follow the algorithm for tracking by feature detection discussed in the lecture. You can use cv2.SIFT_create(), sift.detectAndCompute,cv2.BFMatcher(), cv2.drawMatches() function here to compare. 


In [ ]:
import cv2
import numpy as np

class ImprovedTemplateMatcher:
    def __init__(self):
        pass

    def multi_scale_matching(self, frame, template):
        """
        Perform multi-scale template matching to handle scale changes.
        """
        ### ENTER YOUR CODE HERE ###
        # Resize the template to different scales 
        pass

    def rotation_invariant_matching(self, frame, template):
        """
        Perform rotation-invariant template matching.
        """
        ### ENTER YOUR CODE HERE ###
        # Rotate the template at different angles 
        pass

    def feature_based_matching(self, frame, template):
        """
        Use SIFT feature detection for robust template matching.
        """
        ### ENTER YOUR CODE HERE ###
        # Use cv2.SIFT_create(), detectAndCompute() to extract features.
        # Match features using cv2.BFMatcher() and visualize with cv2.drawMatches().
        pass

    def draw_bounding_box(self, frame, top_left, template_size):
        """
        Draw bounding box around the detected object.
        """
        ### ENTER YOUR CODE HERE ###
        # Use cv2.rectangle() to draw the bounding box.
        pass

# Load Video
cap = cv2.VideoCapture('input_video.mp4')  # Replace with your video file

### ENTER YOUR CODE TO PROCESS THE VIDEO HERE ###


cap.release()
cv2.destroyAllWindows()
